
# AI Model Intake Identity Card
### OWASP AIBOM-inspired metadata triage for Hugging Face models

This notebook is designed for a **security quarantine / model intake gateway**.

It does **not load or execute the model**. It only retrieves repository metadata and small text/JSON metadata files from Hugging Face, then produces a normalized **Model Identity Card** containing:

- Model name and repository owner
- Model family / subtype
- Pipeline task / intended purpose
- Input and output modalities
- Architecture and base model
- Library / framework
- License, datasets, languages
- Parameter count (when discoverable)
- Commit SHA and repository provenance
- File / serialization surface
- Custom-code indicators
- Hugging Face security-scan metadata when available
- Comparable models that perform the same Hugging Face task
- A recommended security test route based on model family
- An OWASP AIBOM field-mapping bridge

## Security boundary

This notebook intentionally avoids:

- `AutoModel.from_pretrained(...)`
- `torch.load(...)`
- `pickle.load(...)`
- `trust_remote_code=True`
- importing Python files from the model repository
- executing model inference

The intake-card stage should remain **metadata-only**. Artifact scanning and behavioral testing should happen later inside the quarantine environment.

## Why not just run the full OWASP AIBOM Generator?

The OWASP AIBOM Generator is excellent for standards-aligned CycloneDX AIBOM generation and supply-chain transparency. This notebook is deliberately lighter: it adds **model-family classification, comparable-model discovery, and security-test routing** without requiring a full model/runtime stack.

You can later connect this output to the OWASP AIBOM Generator if you need a formal CycloneDX AIBOM.


In [1]:

# Colab setup
# Metadata-only dependencies. We intentionally do NOT install torch/transformers.
!pip -q install -U "huggingface_hub>=0.26" pandas pyyaml requests


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.9/842.9 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.



## 1. Configuration

For public Hugging Face repositories, `HF_TOKEN` can remain unset.

For gated/private repositories:
1. Create a Hugging Face token with the minimum permissions required.
2. In Colab, store it in **Secrets** as `HF_TOKEN`, or set it through an environment variable.
3. Do not hard-code production tokens into the notebook.

### Production recommendation
At the enterprise gateway, set `REQUIRE_EXACT_MODEL_ID = True`.

That forces requests to use an exact `organization/model` identifier and prevents an ambiguous short name from silently resolving to the wrong repository.


In [13]:

import os
import re
import json
import math
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
from IPython.display import display, Markdown
from huggingface_hub import HfApi, ModelCard, hf_hub_download
from huggingface_hub.utils import (
    HfHubHTTPError,
    RepositoryNotFoundError,
    RevisionNotFoundError,
    EntryNotFoundError,
)

# ---------------------------------------------------------------------------
# USER / DEPLOYMENT CONFIGURATION
# ---------------------------------------------------------------------------

HF_TOKEN = os.getenv("HF_TOKEN") or None

# False is convenient for Colab demos because a short name can be searched.
# IMPORTANT: Set this to True at an enterprise intake gateway.
REQUIRE_EXACT_MODEL_ID = True

# Number of same-task candidate alternatives to return.
ALTERNATIVE_LIMIT = 7

# Keep model-card excerpts short because the README is untrusted external text.
DESCRIPTION_MAX_CHARS = 600

# Folder for exported identity cards.
OUTPUT_DIR = Path("/content/model_identity_cards")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional revision. Use None for repository HEAD.
DEFAULT_REVISION = None

api = HfApi(token=HF_TOKEN)

print("Configuration loaded.")
print(f"REQUIRE_EXACT_MODEL_ID = {REQUIRE_EXACT_MODEL_ID}")
print(f"ALTERNATIVE_LIMIT      = {ALTERNATIVE_LIMIT}")
print(f"OUTPUT_DIR             = {OUTPUT_DIR}")


Configuration loaded.
REQUIRE_EXACT_MODEL_ID = True
ALTERNATIVE_LIMIT      = 7
OUTPUT_DIR             = /content/model_identity_cards



## 2. Task / modality catalog

Hugging Face's `pipeline_tag` is one of the strongest standardized signals for identifying what a model is intended to do.

The catalog below maps common tasks to:

- broad security-relevant family,
- model subtype,
- input modality,
- output modality,
- plain-English purpose.

This mapping is intentionally configuration-driven so your security team can extend it without changing the extraction engine.


In [14]:

TASK_CATALOG: Dict[str, Dict[str, Any]] = {
    # Generative language
    "text-generation": {
        "family": "LLM",
        "subtype": "Generative language model",
        "input": ["text"],
        "output": ["text"],
        "purpose": "Generate or continue text from textual prompts.",
    },
    "conversational": {
        "family": "LLM",
        "subtype": "Conversational language model",
        "input": ["text"],
        "output": ["text"],
        "purpose": "Generate conversational responses from textual dialogue.",
    },

    # Vision-language / document intelligence
    "image-text-to-text": {
        "family": "VLM",
        "subtype": "Vision-language generative model",
        "input": ["image", "text"],
        "output": ["text"],
        "purpose": "Reason over images and optional text, then generate text.",
    },
    "visual-question-answering": {
        "family": "VLM",
        "subtype": "Visual question answering",
        "input": ["image", "text"],
        "output": ["text"],
        "purpose": "Answer natural-language questions about image content.",
    },
    "document-question-answering": {
        "family": "OCR / Document AI",
        "subtype": "Document understanding",
        "input": ["document/image", "text"],
        "output": ["text"],
        "purpose": "Extract and reason over textual/visual information in documents.",
    },
    "image-to-text": {
        "family": "VLM",
        "subtype": "Image-to-text",
        "input": ["image"],
        "output": ["text"],
        "purpose": "Convert visual content into textual output such as captions or recognized text.",
    },

    # Audio
    "automatic-speech-recognition": {
        "family": "Audio",
        "subtype": "Automatic speech recognition",
        "input": ["audio"],
        "output": ["text"],
        "purpose": "Transcribe spoken audio into text.",
    },
    "audio-classification": {
        "family": "Audio",
        "subtype": "Audio classifier",
        "input": ["audio"],
        "output": ["label/score"],
        "purpose": "Classify audio into one or more categories.",
    },
    "text-to-speech": {
        "family": "Audio",
        "subtype": "Text-to-speech",
        "input": ["text"],
        "output": ["audio"],
        "purpose": "Synthesize speech or other audio from text.",
    },
    "audio-to-audio": {
        "family": "Audio",
        "subtype": "Audio transformation",
        "input": ["audio"],
        "output": ["audio"],
        "purpose": "Transform an audio signal into another audio signal.",
    },

    # Vision
    "image-classification": {
        "family": "Computer Vision",
        "subtype": "Image classifier",
        "input": ["image"],
        "output": ["label/score"],
        "purpose": "Classify images into one or more categories.",
    },
    "object-detection": {
        "family": "Computer Vision",
        "subtype": "Object detector",
        "input": ["image"],
        "output": ["bounding boxes", "labels", "scores"],
        "purpose": "Detect and localize objects in images.",
    },
    "image-segmentation": {
        "family": "Computer Vision",
        "subtype": "Image segmentation",
        "input": ["image"],
        "output": ["segmentation mask"],
        "purpose": "Partition image regions into semantic or instance-level masks.",
    },
    "depth-estimation": {
        "family": "Computer Vision",
        "subtype": "Depth estimation",
        "input": ["image"],
        "output": ["depth map"],
        "purpose": "Estimate scene depth from image input.",
    },

    # Image generation / diffusion
    "text-to-image": {
        "family": "Generative Vision",
        "subtype": "Text-to-image",
        "input": ["text"],
        "output": ["image"],
        "purpose": "Generate images from textual prompts.",
    },
    "image-to-image": {
        "family": "Generative Vision",
        "subtype": "Image-to-image",
        "input": ["image", "text (optional)"],
        "output": ["image"],
        "purpose": "Generate or transform images from image and optional text input.",
    },
    "unconditional-image-generation": {
        "family": "Generative Vision",
        "subtype": "Unconditional image generation",
        "input": ["latent/noise"],
        "output": ["image"],
        "purpose": "Generate images without a required text prompt.",
    },

    # Embeddings / ranking
    "feature-extraction": {
        "family": "Embedding",
        "subtype": "Feature / embedding model",
        "input": ["text/image/audio depending on model"],
        "output": ["vector/embedding"],
        "purpose": "Convert input data into vector representations.",
    },
    "sentence-similarity": {
        "family": "Embedding",
        "subtype": "Sentence embedding / similarity",
        "input": ["text"],
        "output": ["embedding/similarity score"],
        "purpose": "Represent and compare semantic similarity between text inputs.",
    },

    # Encoder / discriminative NLP
    "text-classification": {
        "family": "NLP",
        "subtype": "Text classifier",
        "input": ["text"],
        "output": ["label/score"],
        "purpose": "Classify textual input.",
    },
    "token-classification": {
        "family": "NLP",
        "subtype": "Token classifier",
        "input": ["text"],
        "output": ["token labels"],
        "purpose": "Assign labels to tokens, such as named-entity recognition.",
    },
    "question-answering": {
        "family": "NLP",
        "subtype": "Extractive question answering",
        "input": ["text", "question"],
        "output": ["text span"],
        "purpose": "Answer questions from supplied textual context.",
    },
    "fill-mask": {
        "family": "NLP",
        "subtype": "Masked language model",
        "input": ["text"],
        "output": ["token probabilities/text"],
        "purpose": "Predict masked tokens in text.",
    },
    "summarization": {
        "family": "NLP",
        "subtype": "Sequence-to-sequence summarization",
        "input": ["text"],
        "output": ["text"],
        "purpose": "Produce a shorter textual summary of source text.",
    },
    "translation": {
        "family": "NLP",
        "subtype": "Machine translation",
        "input": ["text"],
        "output": ["text"],
        "purpose": "Translate text between languages.",
    },
    "zero-shot-classification": {
        "family": "NLP",
        "subtype": "Zero-shot text classifier",
        "input": ["text", "candidate labels"],
        "output": ["labels/scores"],
        "purpose": "Classify text against user-provided candidate labels without task-specific training.",
    },
}

# Heuristics used when pipeline_tag is missing or too broad.
FAMILY_KEYWORDS = {
    "OCR / Document AI": [
        "ocr", "trocr", "donut", "document-ai", "document ai",
        "layoutlm", "nougat", "paddleocr", "text-recognition"
    ],
    "VLM": [
        "vlm", "vision-language", "vision language", "llava", "qwen-vl",
        "qwen2-vl", "qwen3-vl", "idefics", "paligemma", "molmo"
    ],
    "Audio": [
        "whisper", "wav2vec", "speech", "asr", "audio", "tts", "voice"
    ],
    "Embedding": [
        "embedding", "embed", "sentence-transformer", "e5-", "bge-", "gte-"
    ],
    "Reranker": [
        "reranker", "re-ranker", "cross-encoder", "cross encoder"
    ],
    "Generative Vision": [
        "diffusion", "stable-diffusion", "flux", "text-to-image", "sdxl"
    ],
    "Computer Vision": [
        "yolo", "detr", "segmentation", "resnet", "vit-", "convnext"
    ],
    "LLM": [
        "llama", "mistral", "qwen", "gemma", "phi-", "deepseek",
        "falcon", "mixtral", "instruct", "chat"
    ],
}

def _norm_list(value: Any) -> List[Any]:
    if value is None:
        return []
    if isinstance(value, (list, tuple, set)):
        return list(value)
    return [value]

def classify_model_family(
    model_id: str,
    pipeline_tag: Optional[str],
    tags: Optional[List[str]] = None,
    architectures: Optional[List[str]] = None,
) -> Dict[str, Any]:
    tags = tags or []
    architectures = architectures or []

    evidence = []

    # Strongest signal: explicit pipeline task.
    if pipeline_tag in TASK_CATALOG:
        spec = dict(TASK_CATALOG[pipeline_tag])
        evidence.append(f"pipeline_tag={pipeline_tag}")

        # Distinguish OCR from generic image-to-text when repo signals are strong.
        blob = " ".join(
            [model_id, pipeline_tag or "", *tags, *architectures]
        ).lower()
        if pipeline_tag == "image-to-text":
            if any(k in blob for k in FAMILY_KEYWORDS["OCR / Document AI"]):
                spec["family"] = "OCR / Document AI"
                spec["subtype"] = "Optical character recognition / document text extraction"
                spec["purpose"] = "Extract textual information from image or document input."
                evidence.append("OCR keyword/architecture heuristic")

        # Distinguish rerankers from generic text classifiers.
        if pipeline_tag == "text-classification":
            if any(k in blob for k in FAMILY_KEYWORDS["Reranker"]):
                spec["family"] = "Reranker"
                spec["subtype"] = "Cross-encoder / relevance reranker"
                spec["input"] = ["text pair/query-document"]
                spec["output"] = ["relevance score"]
                spec["purpose"] = "Score or rerank candidate text/documents by relevance."
                evidence.append("reranker keyword/architecture heuristic")

        spec["confidence"] = "high"
        spec["evidence"] = evidence
        return spec

    # Fallback: keyword/architecture inference.
    blob = " ".join([model_id, pipeline_tag or "", *tags, *architectures]).lower()
    for family, keywords in FAMILY_KEYWORDS.items():
        if any(k in blob for k in keywords):
            evidence.append(f"keyword/architecture heuristic -> {family}")
            return {
                "family": family,
                "subtype": "Inferred from repository metadata",
                "input": ["unknown"],
                "output": ["unknown"],
                "purpose": f"Purpose could not be reliably standardized; inferred broad family: {family}.",
                "confidence": "medium",
                "evidence": evidence,
            }

    return {
        "family": "Unknown / Manual Review",
        "subtype": "Unclassified",
        "input": ["unknown"],
        "output": ["unknown"],
        "purpose": "Insufficient standardized metadata to infer the model's intended task.",
        "confidence": "low",
        "evidence": ["No recognized pipeline tag or family heuristic matched."],
    }



## 3. Security test-routing policy

This is **routing metadata**, not an approval decision.

The card recommends test categories based on the broad model family. Tool names are deliberately limited to places where the fit is reasonably clear:

- **ModelScan**: static model-serialization scanning for supported formats.
- **garak**: generative LLM vulnerability testing.
- **PyRIT**: configurable generative-AI red-teaming workflows, including text/audio/image/video converters.

For VLM, OCR, audio, vision, embedding, and other model families, the notebook emphasizes the **test objective** rather than pretending that one scanner covers the whole attack surface.


In [4]:

COMMON_TESTS = [
    {
        "layer": "Supply chain / provenance",
        "test": "Pin and verify immutable repository revision (commit SHA) before promotion.",
        "candidate_tools": ["Git/Hugging Face revision pinning", "internal artifact hashing"],
    },
    {
        "layer": "Model artifact",
        "test": "Scan supported serialized model files for unsafe embedded code / deserialization risk.",
        "candidate_tools": ["ModelScan"],
    },
    {
        "layer": "Repository",
        "test": "Review custom Python code, loaders, requirements, and auto_map/remote-code signals before execution.",
        "candidate_tools": ["SAST / dependency scanning / manual code review"],
    },
    {
        "layer": "Governance",
        "test": "Validate license, declared model purpose, training-data metadata, and documentation completeness.",
        "candidate_tools": ["OWASP AIBOM Generator / internal governance controls"],
    },
]

FAMILY_TESTS: Dict[str, List[Dict[str, Any]]] = {
    "LLM": [
        {
            "layer": "Behavioral security",
            "test": "Jailbreak, prompt-injection, data-leakage, system-prompt disclosure and harmful-output evaluation.",
            "candidate_tools": ["garak", "PyRIT"],
        },
        {
            "layer": "Tokenizer / input handling",
            "test": "Exercise Unicode, long-context, delimiter, encoding and malformed-input edge cases.",
            "candidate_tools": ["custom regression corpus"],
        },
    ],
    "VLM": [
        {
            "layer": "Multimodal behavior",
            "test": "Test instructions embedded in images, screenshots, diagrams and image+text conflicts.",
            "candidate_tools": ["PyRIT", "custom multimodal regression corpus"],
        },
        {
            "layer": "Parser / preprocessing",
            "test": "Exercise oversized, malformed, metadata-heavy and adversarial image inputs.",
            "candidate_tools": ["image parser fuzzing / custom corpus"],
        },
    ],
    "OCR / Document AI": [
        {
            "layer": "Document prompt injection",
            "test": "Test hidden/overlaid instructions, tiny text, white-on-white text, rotated text, Unicode and adversarial layouts.",
            "candidate_tools": ["custom document security corpus", "PyRIT where endpoint integration fits"],
        },
        {
            "layer": "Document parser",
            "test": "Exercise malformed/oversized images or documents and decompression/resource-exhaustion cases.",
            "candidate_tools": ["file-format fuzzing / sandbox resource limits"],
        },
    ],
    "Audio": [
        {
            "layer": "Audio behavior",
            "test": "Test spoken/embedded instruction injection, transcription ambiguity, adversarial perturbation and long-input robustness.",
            "candidate_tools": ["custom audio security corpus", "PyRIT converters where applicable"],
        },
        {
            "layer": "Media parser",
            "test": "Exercise malformed containers/codecs, oversized samples and resource-exhaustion behavior.",
            "candidate_tools": ["media fuzzing / sandbox limits"],
        },
    ],
    "Computer Vision": [
        {
            "layer": "Adversarial robustness",
            "test": "Evaluate perturbation, patch/overlay, occlusion, scaling/cropping and label-manipulation robustness.",
            "candidate_tools": ["custom adversarial vision corpus"],
        },
    ],
    "Generative Vision": [
        {
            "layer": "Generation safety",
            "test": "Evaluate prompt abuse, unsafe generation, policy bypass and output metadata leakage.",
            "candidate_tools": ["custom generation safety corpus", "PyRIT where endpoint integration fits"],
        },
    ],
    "Embedding": [
        {
            "layer": "Representation security",
            "test": "Evaluate adversarial semantic collisions, Unicode/normalization edge cases, sensitive-data representation and retrieval manipulation.",
            "candidate_tools": ["custom embedding/RAG security corpus"],
        },
    ],
    "Reranker": [
        {
            "layer": "Ranking integrity",
            "test": "Evaluate adversarial relevance manipulation, query/document injection and ranking instability.",
            "candidate_tools": ["custom ranking security corpus"],
        },
    ],
    "NLP": [
        {
            "layer": "Input / output robustness",
            "test": "Evaluate malformed text, Unicode/normalization, adversarial examples and task-specific abuse cases.",
            "candidate_tools": ["custom NLP security corpus"],
        },
    ],
    "Unknown / Manual Review": [
        {
            "layer": "Classification",
            "test": "Do not auto-route. Manually determine task, modalities, loading mechanism and attack surface first.",
            "candidate_tools": ["manual review"],
        },
    ],
}

def build_test_route(family: str) -> Dict[str, Any]:
    specific = FAMILY_TESTS.get(family, FAMILY_TESTS["Unknown / Manual Review"])
    return {
        "profile_name": f"{family} Security Intake Profile",
        "common_tests": COMMON_TESTS,
        "family_specific_tests": specific,
        "note": (
            "This route is an initial triage profile. Final test selection should also "
            "consider the consuming application, runtime, permissions, tool access, "
            "data sensitivity and deployment architecture."
        ),
    }



## 4. Metadata extraction utilities

Important implementation detail: repository files are **listed**, not executed.

For `README.md` and `config.json`, this notebook downloads only those small metadata files into the local Hugging Face cache. It never downloads model weights as part of identity-card generation.


In [5]:

RISKY_SERIALIZATION_SUFFIXES = (
    ".pkl", ".pickle", ".joblib", ".dill",
    ".pt", ".pth", ".bin", ".ckpt",
    ".h5", ".hdf5", ".keras",
)

LOWER_RISK_WEIGHT_SUFFIXES = (
    ".safetensors", ".gguf", ".onnx", ".tflite",
)

CODE_SUFFIXES = (
    ".py", ".sh", ".bash", ".ps1", ".bat", ".cmd",
    ".js", ".ts", ".so", ".dll", ".dylib",
)

DEPENDENCY_FILES = {
    "requirements.txt", "pyproject.toml", "setup.py", "setup.cfg",
    "environment.yml", "environment.yaml", "Pipfile", "Pipfile.lock",
    "conda.yml", "conda.yaml",
}

def iso_or_none(value: Any) -> Optional[str]:
    if value is None:
        return None
    if hasattr(value, "isoformat"):
        return value.isoformat()
    return str(value)

def object_to_dict(obj: Any) -> Dict[str, Any]:
    if obj is None:
        return {}
    if isinstance(obj, dict):
        return obj
    if hasattr(obj, "to_dict"):
        try:
            return obj.to_dict()
        except Exception:
            pass
    if hasattr(obj, "__dict__"):
        return {
            k: v for k, v in vars(obj).items()
            if not k.startswith("_")
        }
    return {}

def safe_card_data(info: Any, card: Optional[ModelCard]) -> Dict[str, Any]:
    # Prefer explicitly parsed model-card metadata if available.
    if card is not None:
        try:
            return card.data.to_dict()
        except Exception:
            pass
    return object_to_dict(getattr(info, "card_data", None))

def clean_markdown_excerpt(text: Optional[str], max_chars: int = 600) -> Optional[str]:
    if not text:
        return None

    # Drop fenced code blocks.
    text = re.sub(r"```.*?```", " ", text, flags=re.DOTALL)

    # Drop HTML tags.
    text = re.sub(r"<[^>]+>", " ", text)

    # Drop images, simplify links.
    text = re.sub(r"!\[[^\]]*\]\([^)]+\)", " ", text)
    text = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", text)

    # Drop heading markers and repeated whitespace.
    lines = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            lines.append("")
            continue
        line = re.sub(r"^#{1,6}\s+", "", line)
        if line.startswith(("<!--", "---")):
            continue
        lines.append(line)

    text = "\n".join(lines)
    paragraphs = [re.sub(r"\s+", " ", p).strip() for p in re.split(r"\n\s*\n", text)]
    paragraphs = [
        p for p in paragraphs
        if len(p) >= 60 and not p.lower().startswith(("license", "citation"))
    ]

    if not paragraphs:
        return None

    excerpt = paragraphs[0]
    if len(excerpt) > max_chars:
        excerpt = excerpt[:max_chars].rsplit(" ", 1)[0] + "…"
    return excerpt

def get_model_card(model_id: str, revision: Optional[str], token: Optional[str]) -> Tuple[Optional[ModelCard], Optional[str]]:
    """
    Fetch only README.md. No model weights are downloaded.
    """
    try:
        path = hf_hub_download(
            repo_id=model_id,
            filename="README.md",
            revision=revision,
            token=token,
        )
        card = ModelCard.load(path, ignore_metadata_errors=True)
        return card, path
    except (EntryNotFoundError, HfHubHTTPError, OSError, ValueError):
        return None, None

def get_config_json(model_id: str, revision: Optional[str], token: Optional[str]) -> Dict[str, Any]:
    """
    Fetch only config.json if present. This does not import model code.
    """
    try:
        path = hf_hub_download(
            repo_id=model_id,
            filename="config.json",
            revision=revision,
            token=token,
        )
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except (EntryNotFoundError, HfHubHTTPError, OSError, ValueError, json.JSONDecodeError):
        return {}

def fetch_model_info(model_id: str, revision: Optional[str], token: Optional[str]):
    """
    Ask the Hub for repository information, file metadata and security status.
    Compatibility fallback is included for older huggingface_hub versions.
    """
    try:
        return api.model_info(
            repo_id=model_id,
            revision=revision,
            files_metadata=True,
            securityStatus=True,
            token=token,
        )
    except TypeError:
        # Older client versions may not support securityStatus.
        return api.model_info(
            repo_id=model_id,
            revision=revision,
            files_metadata=True,
            token=token,
        )

def extract_parameter_count(info: Any) -> Optional[int]:
    st = getattr(info, "safetensors", None)
    if st is None:
        return None

    # huggingface_hub commonly exposes a dict of dtype -> parameter count.
    params = getattr(st, "parameters", None)
    if params is None and isinstance(st, dict):
        params = st.get("parameters")

    if isinstance(params, dict):
        values = [v for v in params.values() if isinstance(v, (int, float))]
        if values:
            return int(sum(values))

    # Some versions may expose a total-like field.
    for attr in ("total", "total_parameters", "parameter_count"):
        v = getattr(st, attr, None)
        if isinstance(v, (int, float)):
            return int(v)

    return None

def human_count(n: Optional[int]) -> Optional[str]:
    if n is None:
        return None
    for unit, size in [("T", 1_000_000_000_000), ("B", 1_000_000_000), ("M", 1_000_000), ("K", 1_000)]:
        if n >= size:
            return f"{n/size:.2f}{unit}"
    return str(n)

def extract_context_length(config: Dict[str, Any]) -> Optional[int]:
    keys = [
        "max_position_embeddings",
        "n_positions",
        "seq_length",
        "max_sequence_length",
        "model_max_length",
        "context_length",
    ]
    for key in keys:
        value = config.get(key)
        if isinstance(value, int) and value > 0:
            return value

    # Some multimodal/config-composed models nest text_config.
    text_cfg = config.get("text_config")
    if isinstance(text_cfg, dict):
        for key in keys:
            value = text_cfg.get(key)
            if isinstance(value, int) and value > 0:
                return value
    return None

def extract_architectures(config: Dict[str, Any], info: Any) -> List[str]:
    arch = _norm_list(config.get("architectures"))
    if arch:
        return [str(x) for x in arch]

    info_cfg = getattr(info, "config", None)
    if isinstance(info_cfg, dict):
        return [str(x) for x in _norm_list(info_cfg.get("architectures"))]
    return []

def get_repo_files(info: Any) -> List[Dict[str, Any]]:
    files = []
    for sibling in (getattr(info, "siblings", None) or []):
        name = getattr(sibling, "rfilename", None)
        if not name and isinstance(sibling, dict):
            name = sibling.get("rfilename") or sibling.get("path")
        if not name:
            continue

        size = getattr(sibling, "size", None)
        if size is None and isinstance(sibling, dict):
            size = sibling.get("size")

        files.append({
            "name": name,
            "size": size if isinstance(size, int) else None,
        })
    return files

def analyze_artifact_surface(files: List[Dict[str, Any]], config: Dict[str, Any]) -> Dict[str, Any]:
    names = [f["name"] for f in files]
    lower_names = [n.lower() for n in names]

    risky = [n for n in names if n.lower().endswith(RISKY_SERIALIZATION_SUFFIXES)]
    lower_risk = [n for n in names if n.lower().endswith(LOWER_RISK_WEIGHT_SUFFIXES)]
    code_files = [n for n in names if n.lower().endswith(CODE_SUFFIXES)]
    dependency_files = [
        n for n in names
        if Path(n).name in DEPENDENCY_FILES
    ]

    custom_model_code = [
        n for n in code_files
        if Path(n).name.startswith(("modeling_", "configuration_", "tokenization_", "processing_"))
    ]

    auto_map = config.get("auto_map")
    auto_map_signal = bool(auto_map)

    total_bytes = sum(
        f["size"] for f in files
        if isinstance(f.get("size"), int)
    )

    formats = sorted({
        Path(n).suffix.lower()
        for n in names
        if Path(n).suffix
    })

    custom_code_signal = bool(custom_model_code or auto_map_signal)

    return {
        "files_total": len(files),
        "repository_size_bytes_if_reported": total_bytes or None,
        "file_extensions": formats,
        "risky_serialization_files": risky,
        "lower_risk_weight_files": lower_risk,
        "code_files": code_files,
        "dependency_files": dependency_files,
        "custom_model_code_files": custom_model_code,
        "auto_map": auto_map if auto_map_signal else None,
        "custom_code_signal": custom_code_signal,
    }

def get_security_status(info: Any) -> Any:
    # Attribute names have varied between versions.
    for attr in ("security_repo_status", "securityRepoStatus", "security_status"):
        value = getattr(info, attr, None)
        if value is not None:
            if isinstance(value, (str, int, float, bool, list, dict)):
                return value
            return object_to_dict(value)
    return None

def first_present(d: Dict[str, Any], keys: List[str]) -> Any:
    for k in keys:
        value = d.get(k)
        if value not in (None, "", [], {}):
            return value
    return None



## 5. Model ID resolution

The gateway should normally receive `organization/model`.

For Colab convenience, this notebook can also accept:
- a full Hugging Face URL, or
- a short model name.

If a short name is ambiguous, the resolver records the candidate list and marks the resolution as heuristic. In production, disable that behavior with `REQUIRE_EXACT_MODEL_ID = True`.


In [6]:

def normalize_hf_reference(model_ref: str) -> str:
    model_ref = model_ref.strip()
    model_ref = re.sub(r"^https?://huggingface\.co/", "", model_ref, flags=re.I)
    model_ref = model_ref.split("?", 1)[0].split("#", 1)[0].strip("/")
    # Remove common URL suffixes after the repo identifier.
    parts = model_ref.split("/")
    if len(parts) >= 2:
        return "/".join(parts[:2])
    return model_ref

def resolve_model_id(
    model_ref: str,
    token: Optional[str] = None,
    require_exact: bool = False,
) -> Dict[str, Any]:
    normalized = normalize_hf_reference(model_ref)

    # Exact owner/repo input.
    if "/" in normalized:
        return {
            "requested": model_ref,
            "normalized": normalized,
            "resolved_model_id": normalized,
            "resolution": "exact",
            "candidates": [],
        }

    if require_exact:
        raise ValueError(
            "Exact Hugging Face model ID required. Use 'organization/model'."
        )

    # Fuzzy search for notebook/demo convenience only.
    candidates = list(
        api.list_models(
            search=normalized,
            sort="downloads",
            limit=10,
            token=token,
        )
    )

    if not candidates:
        raise RepositoryNotFoundError(
            f"No Hugging Face model candidates found for: {normalized}"
        )

    normalized_lower = normalized.lower()

    # Prefer exact final repo-name match. If multiple, keep the most downloaded
    # because list_models was already sorted that way.
    exact_name_matches = [
        c for c in candidates
        if c.id.split("/")[-1].lower() == normalized_lower
    ]

    chosen = exact_name_matches[0] if exact_name_matches else candidates[0]

    candidate_rows = []
    for c in candidates:
        candidate_rows.append({
            "model_id": c.id,
            "downloads": getattr(c, "downloads", None),
            "likes": getattr(c, "likes", None),
            "pipeline_tag": getattr(c, "pipeline_tag", None),
        })

    return {
        "requested": model_ref,
        "normalized": normalized,
        "resolved_model_id": chosen.id,
        "resolution": "heuristic_search",
        "candidates": candidate_rows,
    }



## 6. Comparable-model discovery

"Alternatives" are **not endorsements** and are not automatically considered safer.

The notebook defines "comparable" as:
1. same Hugging Face `pipeline_tag`,
2. broadly same inferred family,
3. sorted by Hub downloads as a practical discoverability signal.

In an enterprise deployment you can tighten this further by applying:
- approved-license allowlists,
- approved publishers,
- parameter-size ranges,
- architecture constraints,
- only models already scanned/approved by your organization.


In [7]:

def find_alternative_models(
    current_model_id: str,
    pipeline_tag: Optional[str],
    family: str,
    limit: int = 7,
    token: Optional[str] = None,
) -> List[Dict[str, Any]]:
    if not pipeline_tag:
        return []

    # Pull extra candidates because we will exclude the current model
    # and models whose inferred family does not match.
    try:
        candidates = list(
            api.list_models(
                pipeline_tag=pipeline_tag,
                sort="downloads",
                limit=max(limit * 4, 20),
                token=token,
            )
        )
    except TypeError:
        # Compatibility fallback.
        candidates = list(
            api.list_models(
                filter=pipeline_tag,
                sort="downloads",
                limit=max(limit * 4, 20),
                token=token,
            )
        )

    results = []
    for item in candidates:
        if item.id == current_model_id:
            continue

        tags = list(getattr(item, "tags", None) or [])
        alt_pipeline = getattr(item, "pipeline_tag", None) or pipeline_tag
        alt_class = classify_model_family(
            item.id,
            alt_pipeline,
            tags=tags,
            architectures=[],
        )

        if family != "Unknown / Manual Review" and alt_class["family"] != family:
            continue

        results.append({
            "model_id": item.id,
            "organization": getattr(item, "author", None) or item.id.split("/")[0],
            "pipeline_tag": alt_pipeline,
            "family": alt_class["family"],
            "library": getattr(item, "library_name", None),
            "downloads": getattr(item, "downloads", None),
            "likes": getattr(item, "likes", None),
            "url": f"https://huggingface.co/{item.id}",
            "reason": f"Same Hugging Face task: {pipeline_tag}",
        })

        if len(results) >= limit:
            break

    return results



## 7. Generate the Model Identity Card

The single function below is what you would eventually expose at the **model gateway**:

```python
card = generate_model_identity_card("openai/whisper-large-v3")
```

It returns a normal Python dictionary, so it can be:
- returned by an API,
- written to a database,
- attached to a model-intake ticket,
- used to select a downstream security pipeline,
- stored beside the quarantined model artifact.


In [8]:

def generate_model_identity_card(
    model_ref: str,
    revision: Optional[str] = DEFAULT_REVISION,
    token: Optional[str] = HF_TOKEN,
    alternative_limit: int = ALTERNATIVE_LIMIT,
    require_exact_model_id: bool = REQUIRE_EXACT_MODEL_ID,
) -> Dict[str, Any]:

    resolution = resolve_model_id(
        model_ref=model_ref,
        token=token,
        require_exact=require_exact_model_id,
    )
    model_id = resolution["resolved_model_id"]

    info = fetch_model_info(
        model_id=model_id,
        revision=revision,
        token=token,
    )

    # Only small metadata files are fetched.
    card, _ = get_model_card(model_id, revision, token)
    config = get_config_json(model_id, revision, token)

    card_data = safe_card_data(info, card)
    tags = list(getattr(info, "tags", None) or card_data.get("tags") or [])
    pipeline_tag = (
        getattr(info, "pipeline_tag", None)
        or card_data.get("pipeline_tag")
    )

    architectures = extract_architectures(config, info)

    classification = classify_model_family(
        model_id=model_id,
        pipeline_tag=pipeline_tag,
        tags=tags,
        architectures=architectures,
    )

    files = get_repo_files(info)
    artifact_surface = analyze_artifact_surface(files, config)

    parameter_count = extract_parameter_count(info)

    # Card metadata commonly provides these fields.
    license_value = first_present(card_data, ["license", "license_name"])
    languages = _norm_list(first_present(card_data, ["language", "languages"]))
    datasets = _norm_list(first_present(card_data, ["datasets", "dataset"]))
    base_models = _norm_list(
        first_present(card_data, ["base_model", "base_models"])
        or getattr(info, "base_models", None)
    )

    # "Developer" and "shared-by" metadata is not uniformly populated,
    # so repository ownership is preserved separately from declared developer.
    declared_developer = first_present(
        card_data,
        [
            "developers", "developer", "developed_by",
            "model_creator", "model_author", "organization", "shared_by"
        ],
    )

    repo_owner = getattr(info, "author", None) or model_id.split("/")[0]
    repo_name = model_id.split("/")[-1]

    description_excerpt = None
    if card is not None:
        description_excerpt = clean_markdown_excerpt(
            getattr(card, "text", None),
            DESCRIPTION_MAX_CHARS,
        )

    alternatives = find_alternative_models(
        current_model_id=model_id,
        pipeline_tag=pipeline_tag,
        family=classification["family"],
        limit=alternative_limit,
        token=token,
    )

    sha = getattr(info, "sha", None)
    resolved_revision = sha or revision or "main/HEAD"

    warnings = []

    if resolution["resolution"] != "exact":
        warnings.append(
            "Model reference was resolved heuristically from a short name. "
            "Use exact organization/model at the production gateway."
        )

    if not pipeline_tag:
        warnings.append(
            "No standardized pipeline_tag was found; family classification is heuristic."
        )

    if not license_value:
        warnings.append("No license was found in standardized model-card metadata.")

    if card is None:
        warnings.append("README/model card was not available or could not be parsed.")

    if artifact_surface["risky_serialization_files"]:
        warnings.append(
            "Repository contains serialization formats that require static security scanning "
            "before any model load/deserialization."
        )

    if artifact_surface["custom_code_signal"]:
        warnings.append(
            "Custom-code/auto_map signals detected. Do not enable remote code execution "
            "until code review is complete."
        )

    if not sha:
        warnings.append(
            "Immutable commit SHA was not returned; do not promote an unpinned artifact."
        )

    security_status = get_security_status(info)

    identity_card = {
        "schema": {
            "name": "enterprise-ai-model-intake-card",
            "version": "0.1.0",
            "generated_at_utc": datetime.now(timezone.utc).isoformat(),
            "generation_mode": "metadata-only; no model execution",
        },

        "source": {
            "provider": "Hugging Face",
            "requested_reference": model_ref,
            "resolution_method": resolution["resolution"],
            "resolved_model_id": model_id,
            "requested_revision": revision,
            "resolved_commit_sha": sha,
            "resolved_revision": resolved_revision,
            "repository_url": f"https://huggingface.co/{model_id}",
            "resolution_candidates": resolution["candidates"],
        },

        "identity": {
            "name": repo_name,
            "repository_owner": repo_owner,
            "declared_developer": declared_developer,
            "family": classification["family"],
            "subtype": classification["subtype"],
            "classification_confidence": classification["confidence"],
            "classification_evidence": classification["evidence"],
            "pipeline_task": pipeline_tag,
            "purpose": classification["purpose"],
            "input_modalities": classification["input"],
            "output_modalities": classification["output"],
            "architectures": architectures,
            "base_models": base_models,
            "library": getattr(info, "library_name", None) or card_data.get("library_name"),
            "model_type_from_config": config.get("model_type"),
        },

        "model_facts": {
            "license": license_value,
            "languages": languages,
            "datasets": datasets,
            "parameter_count": parameter_count,
            "parameter_count_human": human_count(parameter_count),
            "context_length_if_discoverable": extract_context_length(config),
            "created_at": iso_or_none(getattr(info, "created_at", None)),
            "last_modified": iso_or_none(getattr(info, "last_modified", None)),
            "downloads": getattr(info, "downloads", None),
            "likes": getattr(info, "likes", None),
            "gated": getattr(info, "gated", None),
            "private": getattr(info, "private", None),
            "disabled": getattr(info, "disabled", None),
            "tags": tags,
        },

        "documentation": {
            "model_card_present": card is not None,
            "description_excerpt_untrusted_text": description_excerpt,
            "card_metadata": {
                "license": license_value,
                "languages": languages,
                "datasets": datasets,
                "base_models": base_models,
                "pipeline_tag": pipeline_tag,
            },
        },

        "artifact_surface": {
            **artifact_surface,
            "hugging_face_security_status": security_status,
            "note": (
                "Hub scanner status is an upstream signal, not an organizational approval. "
                "Run your own quarantine controls before loading the artifact."
            ),
        },

        "alternatives": alternatives,

        "security_test_route": build_test_route(classification["family"]),

        # Bridge only: this is NOT itself a complete/validated CycloneDX AIBOM.
        "owasp_aibom_mapping_bridge": {
            "primaryPurpose": pipeline_tag or classification["purpose"],
            "suppliedBy": repo_owner,
            "type": "machine-learning-model",
            "typeOfModel": classification["family"],
            "component_name": repo_name,
            "component_version": sha,
            "downloadLocation": f"https://huggingface.co/{model_id}",
            "description": description_excerpt,
            "licenses": license_value,
            "datasets": datasets,
            "model_task": pipeline_tag,
            "note": (
                "These fields are provided to ease later integration with the OWASP AIBOM "
                "Generator. Use the OWASP generator / CycloneDX library for a standards-valid BOM."
            ),
        },

        "warnings": warnings,
    }

    return identity_card



## 8. Human-readable display + export

The generated JSON remains the source of truth.

The display function produces an analyst-friendly summary for quick intake review.


In [9]:

def display_identity_card(card: Dict[str, Any]) -> None:
    ident = card["identity"]
    facts = card["model_facts"]
    src = card["source"]
    surface = card["artifact_surface"]

    display(Markdown(
        f"""
# 🪪 AI Model Identity Card

**Model:** `{src['resolved_model_id']}`
**Commit SHA:** `{src.get('resolved_commit_sha') or 'Not available'}`
**Family:** **{ident['family']}**
**Subtype:** {ident['subtype']}
**Pipeline task:** `{ident.get('pipeline_task') or 'Unknown'}`
**Repository owner:** `{ident.get('repository_owner') or 'Unknown'}`
**License:** `{facts.get('license') or 'Not declared'}`
**Library:** `{ident.get('library') or 'Unknown'}`
**Parameters:** `{facts.get('parameter_count_human') or 'Not discoverable from metadata'}`
**Input:** {', '.join(ident.get('input_modalities') or ['unknown'])}
**Output:** {', '.join(ident.get('output_modalities') or ['unknown'])}

### Purpose
{ident.get('purpose') or 'Unknown'}

### Classification evidence
{'; '.join(ident.get('classification_evidence') or [])}
"""
    ))

    key_rows = [
        ("Architecture", ", ".join(ident.get("architectures") or []) or "Unknown"),
        ("Base model(s)", ", ".join(map(str, ident.get("base_models") or [])) or "None declared"),
        ("Context length", facts.get("context_length_if_discoverable") or "Unknown"),
        ("Created", facts.get("created_at") or "Unknown"),
        ("Last modified", facts.get("last_modified") or "Unknown"),
        ("Gated", facts.get("gated")),
        ("Private", facts.get("private")),
        ("Downloads", facts.get("downloads")),
        ("Likes", facts.get("likes")),
        ("Model card present", card["documentation"]["model_card_present"]),
        ("Custom code signal", surface.get("custom_code_signal")),
        ("Risky serialization files", len(surface.get("risky_serialization_files") or [])),
        ("Lower-risk weight files", len(surface.get("lower_risk_weight_files") or [])),
    ]
    display(pd.DataFrame(key_rows, columns=["Field", "Value"]))

    excerpt = card["documentation"].get("description_excerpt_untrusted_text")
    if excerpt:
        display(Markdown("### Model-card description excerpt (untrusted external text)"))
        display(Markdown(f"> {excerpt}"))

    if card.get("warnings"):
        display(Markdown("### ⚠️ Intake warnings"))
        for w in card["warnings"]:
            display(Markdown(f"- {w}"))

    if card.get("alternatives"):
        display(Markdown("### Comparable models with the same task"))
        display(pd.DataFrame(card["alternatives"]))

    display(Markdown("### Recommended security test route"))
    route = card["security_test_route"]
    route_rows = []
    for item in route["common_tests"] + route["family_specific_tests"]:
        route_rows.append({
            "Layer": item["layer"],
            "Test objective": item["test"],
            "Candidate tooling": ", ".join(item["candidate_tools"]),
        })
    display(pd.DataFrame(route_rows))

def export_identity_card(
    card: Dict[str, Any],
    output_dir: Path = OUTPUT_DIR,
) -> Dict[str, str]:
    output_dir.mkdir(parents=True, exist_ok=True)

    model_id = card["source"]["resolved_model_id"]
    sha = card["source"].get("resolved_commit_sha") or "unversioned"
    safe_name = re.sub(r"[^A-Za-z0-9._-]+", "_", model_id)

    json_path = output_dir / f"{safe_name}__{sha[:12]}__identity_card.json"

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(card, f, indent=2, ensure_ascii=False, default=str)

    result = {"json": str(json_path)}

    if card.get("alternatives"):
        csv_path = output_dir / f"{safe_name}__{sha[:12]}__alternatives.csv"
        pd.DataFrame(card["alternatives"]).to_csv(csv_path, index=False)
        result["alternatives_csv"] = str(csv_path)

    return result



## 9. Run it

Change only `MODEL_INPUT`.

Recommended test models for validating different families:

- `openai/whisper-large-v3` → audio / ASR
- `Qwen/Qwen2.5-VL-3B-Instruct` → vision-language model
- `microsoft/trocr-base-printed` → OCR
- `sentence-transformers/all-MiniLM-L6-v2` → embedding
- `google-bert/bert-base-uncased` → NLP encoder
- a text-generation model available to your organization → LLM

For a production gateway, use the exact `organization/model` plus an explicit immutable revision whenever the requester supplies one.


In [15]:

# ---------------------------------------------------------------------------
# CHANGE THIS INPUT
# ---------------------------------------------------------------------------

MODEL_INPUT = "openai/whisper-large-v3"
REVISION = None   # e.g. an exact commit SHA if your intake request supplies one

card = generate_model_identity_card(
    model_ref=MODEL_INPUT,
    revision=REVISION,
)

display_identity_card(card)

paths = export_identity_card(card)
print("\nExported:")
for kind, path in paths.items():
    print(f"  {kind}: {path}")



# 🪪 AI Model Identity Card

**Model:** `openai/whisper-large-v3`  
**Commit SHA:** `06f233fe06e710322aca913c1bc4249a0d71fce1`  
**Family:** **Audio**  
**Subtype:** Automatic speech recognition  
**Pipeline task:** `automatic-speech-recognition`  
**Repository owner:** `openai`  
**License:** `apache-2.0`  
**Library:** `transformers`  
**Parameters:** `1.54B`  
**Input:** audio  
**Output:** text  

### Purpose
Transcribe spoken audio into text.

### Classification evidence
pipeline_tag=automatic-speech-recognition


,Field,Value
0,Architecture,WhisperForConditionalGeneration
1,Base model(s),None declared
2,Context length,Unknown
3,Created,2023-11-07T18:41:14+00:00
4,Last modified,2024-08-12T10:20:10+00:00
5,Gated,False
6,Private,False
7,Downloads,4677524
8,Likes,6336
9,Model card present,True


### Model-card description excerpt (untrusted external text)

> Whisper is a state-of-the-art model for automatic speech recognition (ASR) and speech translation, proposed in the paper Robust Speech Recognition via Large-Scale Weak Supervision by Alec Radford et al. from OpenAI. Trained on >5M hours of labeled data, Whisper demonstrates a strong ability to generalise to many datasets and domains in a zero-shot setting.

### ⚠️ Intake warnings

...

### Comparable models with the same task

,model_id,organization,pipeline_tag,family,library,downloads,likes,url,reason
0,jonatasgrosman/wav2vec2-large-xlsr-53-japanese,jonatasgrosman,automatic-speech-recognition,Audio,transformers,17477279,87,https://huggingface.co/jonatasgrosman/wav2vec2...,Same Hugging Face task: automatic-speech-recog...
1,argmaxinc/whisperkit-coreml,argmaxinc,automatic-speech-recognition,Audio,whisperkit,10996186,229,https://huggingface.co/argmaxinc/whisperkit-co...,Same Hugging Face task: automatic-speech-recog...
2,pyannote/speaker-diarization-3.1,pyannote,automatic-speech-recognition,Audio,pyannote-audio,7704093,3841,https://huggingface.co/pyannote/speaker-diariz...,Same Hugging Face task: automatic-speech-recog...
3,openai/whisper-large-v3-turbo,openai,automatic-speech-recognition,Audio,transformers,6637070,3384,https://huggingface.co/openai/whisper-large-v3...,Same Hugging Face task: automatic-speech-recog...
4,jonatasgrosman/wav2vec2-large-xlsr-53-portuguese,jonatasgrosman,automatic-speech-recognition,Audio,transformers,5928434,58,https://huggingface.co/jonatasgrosman/wav2vec2...,Same Hugging Face task: automatic-speech-recog...
5,pyannote/speaker-diarization-community-1,pyannote,automatic-speech-recognition,Audio,pyannote-audio,5437063,2009,https://huggingface.co/pyannote/speaker-diariz...,Same Hugging Face task: automatic-speech-recog...
6,jonatasgrosman/wav2vec2-large-xlsr-53-russian,jonatasgrosman,automatic-speech-recognition,Audio,transformers,4198740,77,https://huggingface.co/jonatasgrosman/wav2vec2...,Same Hugging Face task: automatic-speech-recog...


### Recommended security test route

,Layer,Test objective,Candidate tooling
0,Supply chain / provenance,Pin and verify immutable repository revision (...,"Git/Hugging Face revision pinning, internal ar..."
1,Model artifact,Scan supported serialized model files for unsa...,ModelScan
2,Repository,"Review custom Python code, loaders, requiremen...",SAST / dependency scanning / manual code review
3,Governance,"Validate license, declared model purpose, trai...",OWASP AIBOM Generator / internal governance co...
4,Audio behavior,"Test spoken/embedded instruction injection, tr...","custom audio security corpus, PyRIT converters..."
5,Media parser,"Exercise malformed containers/codecs, oversize...",media fuzzing / sandbox limits



Exported:
  json: /content/model_identity_cards/openai_whisper-large-v3__06f233fe06e7__identity_card.json
  alternatives_csv: /content/model_identity_cards/openai_whisper-large-v3__06f233fe06e7__alternatives.csv



## 10. Gateway / API integration

Once the notebook logic is validated, the gateway does not need notebook UI code.

The important production entry point is:

```python
generate_model_identity_card(model_ref, revision)
```

A model-request gateway can call it before downloading weights.

### Recommended event flow

```text
Model request
    |
    v
[Model Gateway]
    |
    | exact HF repo + optional revision
    v
[Identity Card Service]
    |
    |-- fetch Hub metadata only
    |-- classify model family
    |-- discover same-task alternatives
    |-- detect custom-code / serialization surface
    |-- assign test profile
    v
[Identity Card JSON]
    |
    +--------------------+
    |                    |
    v                    v
[Policy Gate]       [Quarantine Download]
                         |
                         v
                   [Static Artifact Scan]
                         |
                         v
                   [Family-specific Tests]
                         |
                         v
                    [Security Decision]
```

### Production hardening

- Set `REQUIRE_EXACT_MODEL_ID = True`.
- Pin the final download to `resolved_commit_sha`.
- Use a read-only/minimal Hugging Face token where possible.
- Restrict outbound network access from the metadata service.
- Store generated cards in an immutable/auditable database.
- Keep policy (approved publishers/licenses/tasks) separate from extraction.
- Treat model-card text as untrusted content.
- Never let the metadata service execute repository Python code.


In [16]:

# Example gateway-style wrapper.
# Your API framework can call this function directly.

def handle_model_gateway_request(payload: Dict[str, Any]) -> Dict[str, Any]:
    """
    Expected payload:
    {
        "model": "organization/model",
        "revision": "optional commit/tag/branch"
    }
    """
    model_ref = payload["model"]
    revision = payload.get("revision")

    return generate_model_identity_card(
        model_ref=model_ref,
        revision=revision,
        require_exact_model_id=True,  # production-safe default
    )

# Example:
# response = handle_model_gateway_request({
#     "model": "openai/whisper-large-v3",
#     "revision": None,
# })
# print(json.dumps(response, indent=2, default=str))



## 11. Optional policy gate example

The extractor should answer **what the model is**.

A separate policy layer should answer **whether your organization allows it**.

This separation is important because policy changes more often than metadata extraction.


In [17]:

# EXAMPLE ONLY — replace with your organization's real policy.
POLICY = {
    "require_license": True,
    "deny_risky_serialization_without_scan": True,
    "deny_custom_code_without_review": True,
    "require_commit_sha": True,
}

def evaluate_intake_policy(card: Dict[str, Any], policy: Dict[str, Any] = POLICY) -> Dict[str, Any]:
    failures = []
    review = []

    facts = card["model_facts"]
    source = card["source"]
    surface = card["artifact_surface"]

    if policy.get("require_license") and not facts.get("license"):
        failures.append("License metadata missing.")

    if policy.get("require_commit_sha") and not source.get("resolved_commit_sha"):
        failures.append("Immutable commit SHA missing.")

    if (
        policy.get("deny_risky_serialization_without_scan")
        and surface.get("risky_serialization_files")
    ):
        review.append(
            "Serialized executable-capable format detected: require static artifact scan "
            "before any loading/deserialization."
        )

    if (
        policy.get("deny_custom_code_without_review")
        and surface.get("custom_code_signal")
    ):
        review.append("Custom code / auto_map signal detected: require code review.")

    decision = "REJECT_METADATA" if failures else ("REVIEW_REQUIRED" if review else "READY_FOR_QUARANTINE_SCAN")

    return {
        "decision": decision,
        "failures": failures,
        "manual_review": review,
    }

# Example:
# policy_result = evaluate_intake_policy(card)
# display(policy_result)



## 12. Important limitations

This card is intentionally conservative.

1. **Repository owner is not always the original model developer.**  
   The card stores both repository ownership and any developer metadata discoverable in the model card.

2. **A same-task alternative is not automatically safer or better.**  
   The alternatives section is discovery assistance only.

3. **Hub metadata is publisher-controlled.**  
   Treat it as useful evidence, not independently verified truth.

4. **Family classification can be wrong when metadata is poor.**  
   The card includes confidence and evidence so uncertain classifications can be routed to manual review.

5. **Hugging Face security scanner status is not your organization's security approval.**  
   Keep your own quarantine scanning and acceptance gates.

6. **Model safety depends on the consuming system.**  
   An LLM with no tools is a different risk from the same LLM connected to privileged tools, RAG data, secrets, or an autonomous agent.

7. **This notebook does not produce a standards-validated CycloneDX AIBOM.**  
   It provides an `owasp_aibom_mapping_bridge`. Use the OWASP AIBOM Generator or CycloneDX libraries when you need the formal BOM artifact.
